# RANDOM FOREST — Prediksi Kelulusan Mahasiswa

Notebook ini membangun model **Random Forest Classifier** untuk memprediksi status kelulusan mahasiswa berdasarkan berbagai fitur akademik.

## 1. Import Library

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay
)

import warnings
warnings.filterwarnings('ignore')

print('Library berhasil diimport!')

Library berhasil diimport!


## 2. Load Dataset

In [2]:
# Jika menggunakan Google Colab, mount drive terlebih dahulu:
# from google.colab import drive
# drive.mount('/content/drive')
# df = pd.read_csv('/content/drive/MyDrive/dataset_kelulusan_mahasiswa.csv')

df = pd.read_csv('dataset_kelulusan_mahasiswa.csv')
print('Dataset berhasil dimuat!')
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'dataset_kelulusan_mahasiswa.csv'

## 3. Eksplorasi Data (EDA)

In [ ]:
# Dimensi dataset
print('Jumlah baris dan kolom:', df.shape)

In [ ]:
# Tipe data tiap kolom
df.info()

In [ ]:
# Statistik deskriptif
df.describe()

In [ ]:
# Cek missing values
print('Missing values per kolom:')
print(df.isnull().sum())

In [ ]:
# Distribusi target (Status Kelulusan)
print('Distribusi Status Kelulusan:')
print(df['Status Kelulusan'].value_counts())
print()
print(df['Status Kelulusan'].value_counts(normalize=True).map(lambda x: f'{x:.1%}'))

plt.figure(figsize=(5, 4))
df['Status Kelulusan'].value_counts().plot(kind='bar', color=['steelblue', 'tomato'], edgecolor='black')
plt.title('Distribusi Status Kelulusan')
plt.xlabel('Status (1=Lulus, 0=Tidak Lulus)')
plt.ylabel('Jumlah Mahasiswa')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Distribusi fitur numerik
numerik = ['IPK', 'IPS Rata-rata', 'IPS Semester Akhir', 'IPS Tren',
           'Mata Kuliah Tidak Lulus', 'Jumlah Cuti Akademik', 'Jumlah Semester']

df[numerik].hist(bins=20, figsize=(14, 8), color='steelblue', edgecolor='black')
plt.suptitle('Distribusi Fitur Numerik', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Korelasi antar fitur numerik
plt.figure(figsize=(9, 6))
corr = df[numerik + ['Status Kelulusan']].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5)
plt.title('Heatmap Korelasi Fitur')
plt.tight_layout()
plt.show()

## 4. Preprocessing Data

In [ ]:
# Encode kolom kategorikal
d_pekerjaan = {'Ya': 1, 'Tidak': 0}
df['Pekerjaan Sambil Kuliah'] = df['Pekerjaan Sambil Kuliah'].map(d_pekerjaan)

d_kehadiran = {'Rendah': 0, 'Sedang': 1, 'Tinggi': 2}
df['Kategori Kehadiran'] = df['Kategori Kehadiran'].map(d_kehadiran)

print('Encoding selesai. Contoh data:')
df.head()

In [ ]:
# Pisahkan fitur (X) dan target (y)
features = [
    'IPK', 'Mata Kuliah Tidak Lulus', 'Jumlah Cuti Akademik',
    'Pekerjaan Sambil Kuliah', 'Jumlah Semester',
    'IPS Rata-rata', 'IPS Semester Akhir', 'IPS Tren',
    'Kategori Kehadiran'
]

X = df[features]
y = df['Status Kelulusan']

print('Jumlah fitur:', X.shape[1])
print('Jumlah data:', X.shape[0])

## 5. Split Data (Train & Test)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,       # 80% train, 20% test
    random_state=42,     # agar hasil dapat direproduksi
    stratify=y           # menjaga proporsi kelas tetap seimbang
)

print(f'Data latih : {X_train.shape[0]} baris')
print(f'Data uji   : {X_test.shape[0]} baris')

## 6. Membuat & Melatih Model Random Forest

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=100,      # jumlah pohon
    max_depth=10,          # batas kedalaman pohon (hindari overfitting)
    min_samples_split=5,   # minimal sampel untuk melakukan split
    min_samples_leaf=2,    # minimal sampel di setiap daun
    criterion='gini',      # ukuran kualitas split: 'gini' atau 'entropy'
    random_state=42
)

rf_model.fit(X_train, y_train)
print('Model berhasil dilatih!')

## 7. Evaluasi Model

In [ ]:
# Prediksi pada data uji
y_pred = rf_model.predict(X_test)

# Akurasi
akurasi = accuracy_score(y_test, y_pred)
print(f'Akurasi Model: {akurasi:.4f} ({akurasi*100:.2f}%)')

In [ ]:
# Classification Report (Precision, Recall, F1-Score)
print('Classification Report:')
print(classification_report(y_test, y_pred, target_names=['Tidak Lulus', 'Lulus']))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Tidak Lulus', 'Lulus'])
disp.plot(cmap='Blues', colorbar=False)
plt.title('Confusion Matrix')
plt.tight_layout()
plt.show()

## 8. Feature Importance

In [ ]:
# Fitur mana yang paling berpengaruh pada prediksi?
importances = rf_model.feature_importances_
feat_df = pd.DataFrame({
    'Fitur': features,
    'Importance': importances
}).sort_values('Importance', ascending=False)

plt.figure(figsize=(9, 5))
sns.barplot(data=feat_df, x='Importance', y='Fitur', palette='viridis')
plt.title('Feature Importance — Random Forest')
plt.xlabel('Tingkat Kepentingan')
plt.tight_layout()
plt.show()

print(feat_df.to_string(index=False))

## 9. Prediksi Data Baru

In [ ]:
# Contoh data mahasiswa baru
# Urutan: IPK, MK Tidak Lulus, Cuti, Pekerjaan, Semester, IPS Rata, IPS Akhir, IPS Tren, Kehadiran
data_baru = pd.DataFrame([{
    'IPK': 3.5,
    'Mata Kuliah Tidak Lulus': 1,
    'Jumlah Cuti Akademik': 0,
    'Pekerjaan Sambil Kuliah': 0,   # 0 = Tidak, 1 = Ya
    'Jumlah Semester': 8,
    'IPS Rata-rata': 3.4,
    'IPS Semester Akhir': 3.6,
    'IPS Tren': 0.3,
    'Kategori Kehadiran': 2          # 0=Rendah, 1=Sedang, 2=Tinggi
}])

# Prediksi
hasil = rf_model.predict(data_baru)[0]
probabilitas = rf_model.predict_proba(data_baru)[0]

label = 'LULUS' if hasil == 1 else 'TIDAK LULUS'
print(f'Hasil Prediksi    : {label}')
print(f'Probabilitas Tidak Lulus : {probabilitas[0]:.2%}')
print(f'Probabilitas Lulus       : {probabilitas[1]:.2%}')